In [2]:
import ee
import geemap
import os

ee.Initialize(project="albaniasat")
print("GEE connected!")

GEE connected!


In [3]:
albania = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017") \
            .filter(ee.Filter.eq("country_na", "Albania"))

print("Albania boundary loaded")

Albania boundary loaded


In [4]:
s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterBounds(albania.geometry()) \
        .filterDate("2021-01-01", "2021-12-31") \
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10)) \
        .select(["B4", "B3", "B2"])

print(f"Images found: {s2.size().getInfo()}")

Images found: 448


In [5]:
# Create one clean composite image from all 448 scenes
composite = s2.median()

# Add NIR band
composite = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                .filterBounds(albania.geometry())
                .filterDate("2021-06-01", "2021-09-30")
                .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
                .select(["B4", "B3", "B2", "B8"])
                .median())

print("Composite image ready!")
info = composite.getInfo()
print("Bands:", [b["id"] for b in info["bands"]])

Composite image ready!
Bands: ['B4', 'B3', 'B2', 'B8']


In [6]:
import math

# Get Albania geometry
albania_geom = albania.geometry()
bounds = albania_geom.bounds().getInfo()["coordinates"][0]

# Albania bounds
min_lon = 19.3
max_lon = 21.1
min_lat = 39.6
max_lat = 42.7

# Each patch = 64 pixels * 10m = 640m ≈ 0.006 degrees
patch_size_deg = 0.006

# Count how many patches we'll have
cols = math.ceil((max_lon - min_lon) / patch_size_deg)
rows = math.ceil((max_lat - min_lat) / patch_size_deg)
print(f"Grid size: {cols} cols x {rows} rows = {cols * rows} total patches")

Grid size: 301 cols x 517 rows = 155617 total patches


155617 patches is way too many to export. We should aim for around 27000, 2000-3000 per class, across 10 classes.

Step 1  →  Load CORINE land cover map

Step 2  →  Define our classes (e.g. Forest, Farmland, Urban...)

Step 3  →  For each class, randomly sample N patch locations

Step 4  →  Export only those patches (labeled and ready)

In [7]:
# Load CORINE Land Cover map
corine = ee.Image("COPERNICUS/CORINE/V20/100m/2018").select("landcover")

# Clip it to Albania only
corine_albania = corine.clip(albania.geometry())

# Check what classes exist in Albania
values = corine_albania.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=albania.geometry(),
    scale=100,
    maxPixels=1e9
).getInfo()

print("Land cover classes found in Albania:")
for code, count in sorted(values["landcover"].items(), key=lambda x: -x[1]):
    print(f"  Class {code}: {int(count)} pixels")

Land cover classes found in Albania:
  Class 311: 577016 pixels
  Class 324: 394805 pixels
  Class 321: 308616 pixels
  Class 323: 285155 pixels
  Class 243: 248968 pixels
  Class 211: 203390 pixels
  Class 242: 195793 pixels
  Class 333: 143038 pixels
  Class 231: 77403 pixels
  Class 312: 74283 pixels
  Class 112: 65517 pixels
  Class 313: 59311 pixels
  Class 223: 48901 pixels
  Class 512: 44968 pixels
  Class 322: 23377 pixels
  Class 332: 19883 pixels
  Class 331: 19643 pixels
  Class 222: 13859 pixels
  Class 212: 12825 pixels
  Class 511: 8382 pixels
  Class 334: 5845 pixels
  Class 421: 5200 pixels
  Class 521: 4639 pixels
  Class 121: 4247 pixels
  Class 411: 4234 pixels
  Class 523: 3659 pixels
  Class 221: 3599 pixels
  Class 131: 3242 pixels
  Class 335: 2293 pixels
  Class 133: 1159 pixels
  Class 124: 1062 pixels
  Class 422: 947 pixels
  Class 111: 663 pixels
  Class 122: 486 pixels
  Class 142: 353 pixels
  Class 141: 254 pixels
  Class 241: 237 pixels
  Class 123: 159 

## CORINE Land Cover Classes Found in Albania

| Code | Class Name                     | Pixels  |
|------|--------------------------------|---------|
| 311  | Broad-leaved forest            | 577,016 |
| 324  | Transitional woodland/shrub    | 394,805 |
| 321  | Natural grassland              | 308,616 |
| 323  | Sclerophyllous vegetation      | 285,155 |
| 243  | Agriculture + natural veg.     | 248,968 |
| 211  | Non-irrigated arable land      | 203,390 |
| 242  | Complex cultivation patterns   | 195,793 |
| 333  | Sparsely vegetated areas       | 143,038 |
| 231  | Pastures                       | 77,403  |
| 312  | Coniferous forest              | 74,283  |
| 112  | Discontinuous urban fabric     | 65,517  |
| 313  | Mixed forest                   | 59,311  |
| 223  | Olive groves                   | 48,901  |
| 512  | Water bodies                   | 44,968  |

## AlbaniaSAT Class Groupings

| AlbaniaSAT Class | CORINE Codes     |
|------------------|------------------|
| Forest           | 311, 312, 313    |
| Shrubland        | 323, 324         |
| Agricultural     | 211, 242, 243    |
| Grassland        | 231, 321         |
| Olive Groves     | 223              |
| Urban            | 112              |
| Water            | 512              |
| Bare/Sparse      | 333              |

1. Forest is too merged

Grouping broad-leaved, coniferous and mixed forest into one class. But spectrally they look quite different to Sentinel-2. Keeping them separate might make the dataset more scientifically rich and give 3 classes out of one group.

2. Urban and Water might not have enough pixels

Urban (65k pixels) and Water (44k pixels) are the smallest classes. At 100m CORINE resolution, it might struggle to sample enough clean 64×64 patches.

3. Olive Groves is your strongest unique contribution

This is the class EuroSAT doesn't have. It's Albania-specific, geographically meaningful, and a genuine argument for why AlbaniaSAT needs to exist. (Hilight for Paper)

4. Shrubland vs Grassland might be spectrally hard to separate

These two could confuse the model. If accuracy suffers, itll likely be here.

## AlbaniaSAT Class Groupings

| AlbaniaSAT Class | CORINE Codes  |
|------------------|---------------|
| Broad-leaved Forest | 311        |
| Coniferous Forest   | 312        |
| Shrubland           | 323, 324   |
| Agricultural        | 211, 242, 243 |
| Grassland           | 231, 321   |
| Olive Groves        | 223        |
| Urban               | 112        |
| Water               | 512        |

1. Spectral Honesty

Broad-leaved and coniferous forests reflect light differently, broad-leaved trees have wide flat leaves, coniferous have thin needles. Sentinel-2 can clearly see this difference, especially in the NIR band. Merging them into one class would be throwing away real signal.

2. A More Meaningful Benchmark

Finer-grained classes make the dataset more detailed than EuroSAT, not less. A model that can tell broad-leaved from coniferous forest is a more capable model than one that just says "forest." That's a better result to publish.

In [ ]:
# Define 8 classes with their CORINE codes
classes = {
    "Broad-leaved Forest": [311],
    "Coniferous Forest": [312],
    "Shrubland": [323, 324],
    "Agricultural":  [211, 242, 243],
    "Grassland": [231, 321],
    "Olive Groves": [223],
    "Urban": [112],
    "Water": [512],
}

# How many patches per class
N_PATCHES = 2000

# Sample patch center points for each class
all_samples = {}

for class_name, codes in classes.items():
    # Build a mask for this class
    mask = corine_albania.eq(codes[0])
    for code in codes[1:]:
        mask = mask.Or(corine_albania.eq(code))
    
    # Sample N random points within this class
    points = (composite
                .updateMask(mask)
                .sample(
                    region=albania.geometry(),
                    scale=640,        # one point per 640m = one patch
                    numPixels=N_PATCHES,
                    seed=42,
                    geometries=True
                ))
    
    count = points.size().getInfo()
    all_samples[class_name] = points
    print(f"{class_name}: {count} sample points found")

Broad-leaved Forest: 403 sample points found
Coniferous Forest: 62 sample points found
Shrubland: 483 sample points found
Agricultural: 440 sample points found
Grassland: 267 sample points found
Olive Groves: 30 sample points found
Urban: 42 sample points found
Water: 22 sample points found


The numbers are lower than expected...

In [9]:
# Define 8 classes with their CORINE codes
classes = {
    "Broad-leaved Forest": [311],
    "Coniferous Forest": [312],
    "Shrubland": [323, 324],
    "Agricultural":  [211, 242, 243],
    "Grassland": [231, 321],
    "Olive Groves": [223],
    "Urban": [112],
    "Water": [512],
}

# How many patches per class
N_PATCHES = 500

# Sample patch center points for each class
all_samples = {}

for class_name, codes in classes.items():
    # Build a mask for this class
    mask = corine_albania.eq(codes[0])
    for code in codes[1:]:
        mask = mask.Or(corine_albania.eq(code))
    
    # Sample N random points within this class
    points = (composite
                .updateMask(mask)
                .sample(
                    region=albania.geometry(),
                    scale=100,        # allow overlapping patches
                    numPixels=N_PATCHES,
                    seed=42,
                    geometries=True
                ))
    
    count = points.size().getInfo()
    all_samples[class_name] = points
    print(f"{class_name}: {count} sample points found")

Broad-leaved Forest: 96 sample points found
Coniferous Forest: 16 sample points found
Shrubland: 134 sample points found
Agricultural: 103 sample points found
Grassland: 71 sample points found
Olive Groves: 7 sample points found
Urban: 8 sample points found
Water: 7 sample points found


In [11]:
N_PATCHES = 500
all_samples = {}

for class_name, codes in classes.items():
    # Build mask
    mask = corine_albania.eq(codes[0])
    for code in codes[1:]:
        mask = mask.Or(corine_albania.eq(code))

    # Sample directly using stratified sampling on the mask
    points = mask.selfMask().stratifiedSample(
        numPoints=N_PATCHES,
        classBand="landcover",
        region=albania.geometry(),
        scale=100,
        seed=42,
        geometries=True
    )

    count = points.size().getInfo()
    all_samples[class_name] = points
    print(f"{class_name}: {count} sample points found")

Broad-leaved Forest: 500 sample points found
Coniferous Forest: 500 sample points found
Shrubland: 500 sample points found
Agricultural: 500 sample points found
Grassland: 500 sample points found
Olive Groves: 500 sample points found
Urban: 500 sample points found
Water: 500 sample points found


In [14]:
for class_name, points in all_samples.items():
    clean_name = class_name.replace(" ", "_")
    
    # Build one image stack for all 500 points in this class
    patches = composite.sampleRegions(
        collection=points,
        scale=10,
        geometries=True
    )
    
    task = ee.batch.Export.table.toDrive(
        collection=patches,
        description=f"AlbaniaSAT_{clean_name}",
        folder="AlbaniaSAT",
        fileNamePrefix=f"AlbaniaSAT_{clean_name}",
        fileFormat="GeoJSON"
    )
    task.start()
    print(f"Queued export for {class_name}")

print("All 8 tasks queued! Check: code.earthengine.google.com/tasks")

Queued export for Broad-leaved Forest
Queued export for Coniferous Forest
Queued export for Shrubland
Queued export for Agricultural
Queued export for Grassland
Queued export for Olive Groves
Queued export for Urban
Queued export for Water
All 8 tasks queued! Check: code.earthengine.google.com/tasks
